# 07 · Ingest the documented OncoPlate benchmark

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

The flagship starts here after the pilot. Actual source records, references, data rights and reviews are needed.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Select the new benchmark without overwriting FoodNExTDB

In [ ]:
from oncoplate.governance import require_gate
require_gate(cfg,'new_collection');require_gate(cfg,'supervisor_execution')
NEW_DATASET_VERSION='oncoplate_v3_main_v1'
cfg['study']['dataset']=NEW_DATASET_VERSION
p=initialize(cfg)
import_root=Path(cfg['root'])/'data/imports'/NEW_DATASET_VERSION
print('Place canonical reviewed inputs in',import_root)

## 2. Validate actual records, focal-item identities and annotations

In [ ]:
from oncoplate.datasets import ingest_documented
records,ann=ingest_documented(import_root/'records.jsonl',import_root/'annotations.jsonl',p['prepared'],allowed_root=import_root)
print(len(records),len(ann))

## 3. Validate inference-only sources and copy reviewed rules

In [ ]:
from oncoplate.inputs import validate_states
states=read_json(import_root/'inference_states.json');rules=read_json(import_root/'claim_rules.json')
print(validate_states(records,states))
write_json(p['private']/"inference_states.json",states);write_json(p['private']/"claim_rules.json",rules)
write_json(p['private']/"protocol_context.json",{'asof':utcnow(),'input_mode':'multimodal','primary_anchor_family':'dinov2_vits14_finetune_joint','scope':'new_documented_benchmark'})

## 4. Activate this version for subsequent notebooks
Revisit notebook 03 for duplicate review, group splitting and fit-only vocabularies. Do not merge public and new records into one random split.

In [ ]:
write_json(Path(cfg['root'])/'governance/active_study.json',{'dataset':NEW_DATASET_VERSION,'activated_at':utcnow()})
print('Activated',NEW_DATASET_VERSION,'Return to notebooks 03–05, then continue to the full grid.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
